In [0]:
from pyspark.sql.types import *
data=[(0,0,'start',0.712),(0,0,'end',1.520),(0,1,'start',3.140),(0,1,'end',4.120),
      (1,0,'start',0.550),(1,0,'end',1.550),(1,1,'start',0.430),(1,1,'end',1.420),
      (2,0,'start',4.100),(2,0,'end',4.512),(2,1,'start',2.500),(2,1,'end',5.000)]
schema=["Machine_id","processid","activityid","timestamp"]
df1=spark.createDataFrame(data,schema)
display(df1)

Machine_id,processid,activityid,timestamp
0,0,start,0.712
0,0,end,1.52
0,1,start,3.14
0,1,end,4.12
1,0,start,0.55
1,0,end,1.55
1,1,start,0.43
1,1,end,1.42
2,0,start,4.1
2,0,end,4.512


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import *

In [0]:
df2 = df1.withColumn("start_time",lag(df1.timestamp).over(Window.orderBy("Machine_id","processid",desc("activityid")))).filter("activityid == 'end'")
df2.display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Machine_id,processid,activityid,timestamp,start_time
0,0,end,1.52,0.712
0,1,end,4.12,3.14
1,0,end,1.55,0.55
1,1,end,1.42,0.43
2,0,end,4.512,4.1
2,1,end,5.0,2.5


In [0]:
df3 = df2.withColumn("duration",df2.timestamp - df2.start_time)
df3.display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Machine_id,processid,activityid,timestamp,start_time,duration
0,0,end,1.52,0.712,0.808
0,1,end,4.12,3.14,0.98
1,0,end,1.55,0.55,1.0
1,1,end,1.42,0.43,0.99
2,0,end,4.512,4.1,0.4119999999999999
2,1,end,5.0,2.5,2.5


In [0]:
df4 = df3.groupBy("Machine_id").agg(avg("duration").alias("avg_processing time"))
df4.display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Machine_id,avg_processing time
0,0.894
1,0.995
2,1.456
